<a href="https://colab.research.google.com/github/Hoor-Fayaz/RehabIQ/blob/main/RehabIQ_LSTM_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# REHABIQ LSTM TRAINING
# STEP 1: MOUNT GOOGLE DRIVE AND VERIFY PATHS
# ============================================================

from google.colab import drive
drive.mount('/content/gdrive')

from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/RehabIQ_Dataset")

MODEL_INPUT_DIR = ROOT / "stage10_6_model_inputs"
METADATA_DIR = ROOT / "metadata"
MODEL_OUTPUT_DIR = ROOT / "models" / "LSTM_Experiments"

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted successfully.")
print("Dataset root:", ROOT)
print("Model inputs:", MODEL_INPUT_DIR)
print("Output folder:", MODEL_OUTPUT_DIR)

assert ROOT.exists(), "RehabIQ dataset folder not found!"
assert MODEL_INPUT_DIR.exists(), "Model input folder not found!"

required_files = [
    "EXP1_baseline_train_X.npy",
    "EXP1_baseline_train_mask.npy",
    "EXP1_baseline_train_y.npy",

    "EXP2_class_weights_train_X.npy",
    "EXP2_class_weights_train_mask.npy",
    "EXP2_class_weights_train_y.npy",

    "EXP3_augmentation_train_X.npy",
    "EXP3_augmentation_train_mask.npy",
    "EXP3_augmentation_train_y.npy",

    "EXP4_combined_train_X.npy",
    "EXP4_combined_train_mask.npy",
    "EXP4_combined_train_y.npy",

    "shared_validation_X.npy",
    "shared_validation_mask.npy",
    "shared_validation_y.npy",

    "shared_test_X.npy",
    "shared_test_mask.npy",
    "shared_test_y.npy"
]

missing = [
    filename for filename in required_files
    if not (MODEL_INPUT_DIR / filename).exists()
]

if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

print("\nPASS: All 18 required model input arrays are present.")

Mounted at /content/gdrive
Google Drive mounted successfully.
Dataset root: /content/gdrive/MyDrive/RehabIQ_Dataset
Model inputs: /content/gdrive/MyDrive/RehabIQ_Dataset/stage10_6_model_inputs
Output folder: /content/gdrive/MyDrive/RehabIQ_Dataset/models/LSTM_Experiments

PASS: All 18 required model input arrays are present.


In [2]:
# ============================================================
# STEP 2: LOAD AND VERIFY REHABIQ MODEL INPUTS
# ============================================================

import numpy as np

# Shared validation data
X_val = np.load(MODEL_INPUT_DIR / "shared_validation_X.npy")
M_val = np.load(MODEL_INPUT_DIR / "shared_validation_mask.npy")
y_val = np.load(MODEL_INPUT_DIR / "shared_validation_y.npy")

# Shared test data
X_test = np.load(MODEL_INPUT_DIR / "shared_test_X.npy")
M_test = np.load(MODEL_INPUT_DIR / "shared_test_mask.npy")
y_test = np.load(MODEL_INPUT_DIR / "shared_test_y.npy")

# Four training experiments
experiments = {
    "EXP1_baseline": {
        "X": np.load(MODEL_INPUT_DIR / "EXP1_baseline_train_X.npy"),
        "mask": np.load(MODEL_INPUT_DIR / "EXP1_baseline_train_mask.npy"),
        "y": np.load(MODEL_INPUT_DIR / "EXP1_baseline_train_y.npy"),
        "use_class_weights": False
    },

    "EXP2_class_weights": {
        "X": np.load(MODEL_INPUT_DIR / "EXP2_class_weights_train_X.npy"),
        "mask": np.load(MODEL_INPUT_DIR / "EXP2_class_weights_train_mask.npy"),
        "y": np.load(MODEL_INPUT_DIR / "EXP2_class_weights_train_y.npy"),
        "use_class_weights": True
    },

    "EXP3_augmentation": {
        "X": np.load(MODEL_INPUT_DIR / "EXP3_augmentation_train_X.npy"),
        "mask": np.load(MODEL_INPUT_DIR / "EXP3_augmentation_train_mask.npy"),
        "y": np.load(MODEL_INPUT_DIR / "EXP3_augmentation_train_y.npy"),
        "use_class_weights": False
    },

    "EXP4_combined": {
        "X": np.load(MODEL_INPUT_DIR / "EXP4_combined_train_X.npy"),
        "mask": np.load(MODEL_INPUT_DIR / "EXP4_combined_train_mask.npy"),
        "y": np.load(MODEL_INPUT_DIR / "EXP4_combined_train_y.npy"),
        "use_class_weights": True
    }
}

# ------------------------------------------------------------
# Validate shared data
# ------------------------------------------------------------

assert X_val.shape == (178, 144, 68)
assert M_val.shape == (178, 144)
assert y_val.shape == (178,)

assert X_test.shape == (192, 144, 68)
assert M_test.shape == (192, 144)
assert y_test.shape == (192,)

assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

# ------------------------------------------------------------
# Validate experiment data
# ------------------------------------------------------------

for name, data in experiments.items():

    X = data["X"]
    mask = data["mask"]
    y = data["y"]

    assert X.ndim == 3 and X.shape[1:] == (144, 68), name
    assert mask.shape == X.shape[:2], name
    assert y.shape == (X.shape[0],), name

    assert np.isfinite(X).all(), f"Non-finite features in {name}"
    assert set(np.unique(y)).issubset({0, 1}), name

    assert set(np.unique(mask)).issubset({False, True}), name

    print(
        f"{name}: "
        f"X={X.shape}, "
        f"mask={mask.shape}, "
        f"y={y.shape}, "
        f"class counts={np.bincount(y.astype(int), minlength=2)}"
    )

print("\nValidation:", X_val.shape, M_val.shape, y_val.shape)
print("Test:", X_test.shape, M_test.shape, y_test.shape)

print("\nPASS: All loaded arrays passed verification.")

EXP1_baseline: X=(788, 144, 68), mask=(788, 144), y=(788,), class counts=[346 442]
EXP2_class_weights: X=(788, 144, 68), mask=(788, 144), y=(788,), class counts=[346 442]
EXP3_augmentation: X=(1576, 144, 68), mask=(1576, 144), y=(1576,), class counts=[692 884]
EXP4_combined: X=(1576, 144, 68), mask=(1576, 144), y=(1576,), class counts=[692 884]

Validation: (178, 144, 68) (178, 144) (178,)
Test: (192, 144, 68) (192, 144) (192,)

PASS: All loaded arrays passed verification.


In [3]:
# ============================================================
# STEP 3: IMPORT TENSORFLOW AND BUILD LSTM ARCHITECTURE
# ============================================================

import tensorflow as tf
import random
import os

from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------------------------------------------
# Class weights calculated during preprocessing
# ------------------------------------------------------------

CLASS_WEIGHTS = {
    0: 1.138728323699422,  # Incorrect
    1: 0.8914027149321267  # Correct
}

print("\nClass weights:")
print(CLASS_WEIGHTS)

# ------------------------------------------------------------
# LSTM architecture
# ------------------------------------------------------------

def build_lstm_model():

    pose_input = Input(
        shape=(144, 68),
        name="pose_sequence"
    )

    mask_input = Input(
        shape=(144,),
        dtype=tf.bool,
        name="valid_mask"
    )

    x = LSTM(
        64,
        return_sequences=True,
        dropout=0.2,
        name="lstm_layer_1"
    )(
        pose_input,
        mask=mask_input
    )

    x = LSTM(
        32,
        return_sequences=False,
        dropout=0.2,
        name="lstm_layer_2"
    )(
        x,
        mask=mask_input
    )

    x = Dropout(0.3)(x)

    output = Dense(
        1,
        activation="sigmoid",
        name="correctness_probability"
    )(x)

    model = Model(
        inputs=[pose_input, mask_input],
        outputs=output,
        name="RehabIQ_LSTM"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model


# Build a sample model and inspect it
sample_model = build_lstm_model()

sample_model.summary()

print("\nPASS: LSTM architecture created.")

TensorFlow version: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Class weights:
{0: 1.138728323699422, 1: 0.8914027149321267}


Model: "RehabIQ_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ pose_sequence       │ (None, 144, 68)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ valid_mask          │ (None, 144)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_layer_1 (LSTM) │ (None, 144, 64)   │     34,048 │ pose_sequence[0]… │
│                     │                   │            │ valid_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_layer_2 (LSTM) │ (None, 32)        │     12,416 │ lstm_layer_1[0][… │
│                     │                   │            │ valid_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ lstm_layer_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ correctness_probab… │ (None, 1)         │         33 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 46,497 (181.63 KB)

 Trainable params: 46,497 (181.63 KB)

 Non-trainable params: 0 (0.00 B)


PASS: LSTM architecture created.


In [4]:
# ============================================================
# STEP 4: TRAIN ALL FOUR REHABIQ LSTM EXPERIMENTS
# ============================================================

import pandas as pd
import json
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

EPOCHS = 60
BATCH_SIZE = 32

all_results = []

for experiment_name, data in experiments.items():

    print("\n" + "=" * 65)
    print("STARTING:", experiment_name)
    print("=" * 65)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    X_train = data["X"].astype(np.float32)
    M_train = data["mask"].astype(bool)
    y_train = data["y"].astype(np.float32)

    # Build a fresh LSTM model
    model = build_lstm_model()

    checkpoint_path = (
        MODEL_OUTPUT_DIR /
        f"{experiment_name}_best.keras"
    )

    # --------------------------------------------------------
    # Callbacks
    # --------------------------------------------------------

    callbacks = [

        ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=1
        ),

        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=0.00001,
            verbose=1
        )
    ]

    # --------------------------------------------------------
    # Training configuration
    # --------------------------------------------------------

    fit_arguments = {}

    if data["use_class_weights"]:
        fit_arguments["class_weight"] = CLASS_WEIGHTS

    print("Training samples:", len(X_train))
    print("Validation samples:", len(X_val))
    print("Class weights enabled:", data["use_class_weights"])

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    history = model.fit(

        x={
            "pose_sequence": X_train,
            "valid_mask": M_train
        },

        y=y_train,

        validation_data=(
            {
                "pose_sequence": X_val,
                "valid_mask": M_val
            },
            y_val.astype(np.float32)
        ),

        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,

        callbacks=callbacks,

        verbose=1,

        **fit_arguments
    )

    # Save training history
    history_df = pd.DataFrame(history.history)

    history_path = (
        MODEL_OUTPUT_DIR /
        f"{experiment_name}_history.csv"
    )

    history_df.to_csv(history_path, index=False)

    # --------------------------------------------------------
    # Load best checkpoint
    # --------------------------------------------------------

    best_model = tf.keras.models.load_model(
        checkpoint_path
    )

    # --------------------------------------------------------
    # Final test evaluation
    # Do not use test results to tune the model.
    # --------------------------------------------------------

    test_probabilities = best_model.predict(
        {
            "pose_sequence": X_test,
            "valid_mask": M_test
        },
        batch_size=BATCH_SIZE,
        verbose=0
    ).ravel()

    test_predictions = (
        test_probabilities >= 0.5
    ).astype(int)

    actual_labels = y_test.astype(int)

    accuracy = accuracy_score(
        actual_labels,
        test_predictions
    )

    precision = precision_score(
        actual_labels,
        test_predictions,
        zero_division=0
    )

    recall = recall_score(
        actual_labels,
        test_predictions,
        zero_division=0
    )

    f1 = f1_score(
        actual_labels,
        test_predictions,
        zero_division=0
    )

    cm = confusion_matrix(
        actual_labels,
        test_predictions,
        labels=[0, 1]
    )

    # --------------------------------------------------------
    # Save experiment results
    # --------------------------------------------------------

    result = {
        "experiment": experiment_name,
        "training_samples": int(len(X_train)),
        "class_weights": bool(data["use_class_weights"]),
        "epochs_completed": int(len(history.history["loss"])),
        "best_validation_loss": float(
            min(history.history["val_loss"])
        ),
        "test_accuracy": float(accuracy),
        "test_precision": float(precision),
        "test_recall": float(recall),
        "test_f1": float(f1),
        "confusion_matrix": cm.tolist(),
        "best_model": str(checkpoint_path)
    }

    all_results.append(result)

    result_path = (
        MODEL_OUTPUT_DIR /
        f"{experiment_name}_results.json"
    )

    with open(result_path, "w") as f:
        json.dump(result, f, indent=4)

    # Save current overall results after every experiment
    with open(
        MODEL_OUTPUT_DIR / "LSTM_all_experiments_results.json",
        "w"
    ) as f:
        json.dump(all_results, f, indent=4)

    print("\nRESULT:", experiment_name)
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1-score:", round(f1, 4))
    print("\nConfusion matrix:")
    print(cm)

    print("\nSaved best model:", checkpoint_path)

    del model
    del best_model

    tf.keras.backend.clear_session()


print("\n" + "=" * 65)
print("ALL FOUR LSTM EXPERIMENTS COMPLETED")
print("=" * 65)


STARTING: EXP1_baseline
Training samples: 788
Validation samples: 178
Class weights enabled: False
Epoch 1/60
24/25 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5500 - auc: 0.5023 - loss: 0.7095 - precision: 0.6337 - recall: 0.6525
Epoch 1: val_loss improved from None to 0.67728, saving model to /content/gdrive/MyDrive/RehabIQ_Dataset/models/LSTM_Experiments/EXP1_baseline_best.keras

Epoch 1: finished saving model to /content/gdrive/MyDrive/RehabIQ_Dataset/models/LSTM_Experiments/EXP1_baseline_best.keras
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.5533 - auc: 0.4954 - loss: 0.7028 - precision: 0.5682 - recall: 0.8484 - val_accuracy: 0.6180 - val_auc: 0.4918 - val_loss: 0.6773 - val_precision: 0.6180 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 2/60
24/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5677 - auc: 0.5380 - loss: 0.6824 - precision: 0.5826 - recall: 0.8533
Epoch 2: val_loss improved from 0.67728 to 0.67605, saving model to /content/gdrive/MyDrive/Reha

In [5]:
# ============================================================
# STEP 5: FINAL LSTM EXPERIMENT COMPARISON
# ============================================================

import pandas as pd
import json

results_file = (
    MODEL_OUTPUT_DIR /
    "LSTM_all_experiments_results.json"
)

with open(results_file, "r") as f:
    results = json.load(f)

comparison = pd.DataFrame(results)

display(
    comparison[
        [
            "experiment",
            "training_samples",
            "class_weights",
            "epochs_completed",
            "best_validation_loss",
            "test_accuracy",
            "test_precision",
            "test_recall",
            "test_f1"
        ]
    ].round(4)
)

comparison.to_csv(
    MODEL_OUTPUT_DIR / "LSTM_Final_Comparison.csv",
    index=False
)

print("\nResults saved in:")
print(MODEL_OUTPUT_DIR)|

SyntaxError: invalid syntax (1834507232.py, line 40)

In [6]:
# ============================================================
# REHABIQ LSTM TRAINING
# STEP 6: FINAL EXPERIMENT COMPARISON
# ============================================================

import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Locate the saved experiment results
# ------------------------------------------------------------

results_file = (
    MODEL_OUTPUT_DIR /
    "LSTM_all_experiments_results.json"
)

# ------------------------------------------------------------
# 2. Check whether training results exist
# ------------------------------------------------------------

if not results_file.exists():

    raise FileNotFoundError(
        "LSTM results file not found. "
        "Please complete the four training experiments first."
    )

# ------------------------------------------------------------
# 3. Load experiment results
# ------------------------------------------------------------

with open(results_file, "r") as f:
    results = json.load(f)

if len(results) == 0:
    raise ValueError("The results file is empty.")

# ------------------------------------------------------------
# 4. Create comparison table
# ------------------------------------------------------------

comparison = pd.DataFrame(results)

required_columns = [
    "experiment",
    "training_samples",
    "class_weights",
    "epochs_completed",
    "best_validation_loss",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1"
]

missing_columns = [
    col for col in required_columns
    if col not in comparison.columns
]

if missing_columns:
    raise ValueError(
        f"Missing result columns: {missing_columns}"
    )

comparison = comparison[required_columns]

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("=" * 70)
print("REHABIQ - LSTM EXPERIMENT COMPARISON")
print("=" * 70)

display(comparison.round(4))

# ------------------------------------------------------------
# 6. Save final comparison CSV
# ------------------------------------------------------------

comparison_path = (
    MODEL_OUTPUT_DIR /
    "LSTM_Final_Comparison.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

# ------------------------------------------------------------
# 7. Identify the experiment with the highest test F1-score
# ------------------------------------------------------------

best_index = comparison["test_f1"].idxmax()
best_experiment = comparison.loc[best_index]

print("\nBEST EXPERIMENT BASED ON TEST F1-SCORE")
print("-" * 50)

print("Experiment:", best_experiment["experiment"])
print("Accuracy:", round(best_experiment["test_accuracy"], 4))
print("Precision:", round(best_experiment["test_precision"], 4))
print("Recall:", round(best_experiment["test_recall"], 4))
print("F1-score:", round(best_experiment["test_f1"], 4))

# ------------------------------------------------------------
# 8. Final save confirmation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPARISON COMPLETED")
print("=" * 70)

print("\nResults saved in:")
print(MODEL_OUTPUT_DIR)

print("\nComparison CSV:")
print(comparison_path)

print("\nAll available LSTM experiment results loaded successfully.")

REHABIQ - LSTM EXPERIMENT COMPARISON


,experiment,training_samples,class_weights,epochs_completed,best_validation_loss,test_accuracy,test_precision,test_recall,test_f1
0,EXP1_baseline,788,False,25,0.6435,0.5990,0.6241,0.7857,0.6957
1,EXP2_class_weights,788,True,18,0.6788,0.5938,0.7179,0.5000,0.5895
2,EXP3_augmentation,1576,False,14,0.6525,0.6042,0.6139,0.8661,0.7185
3,EXP4_combined,1576,True,19,0.6398,0.5885,0.6170,0.7768,0.6877



BEST EXPERIMENT BASED ON TEST F1-SCORE
--------------------------------------------------
Experiment: EXP3_augmentation
Accuracy: 0.6042
Precision: 0.6139
Recall: 0.8661
F1-score: 0.7185

COMPARISON COMPLETED

Results saved in:
/content/gdrive/MyDrive/RehabIQ_Dataset/models/LSTM_Experiments

Comparison CSV:
/content/gdrive/MyDrive/RehabIQ_Dataset/models/LSTM_Experiments/LSTM_Final_Comparison.csv

All available LSTM experiment results loaded successfully.
